# PathSnap — Preprocessing and feature extraction
This notebook processes the available training and student-test trips independently. It creates observation and candidate features for later training and inference; it does **not** create observation-level reference labels. A route is an ordered arc sequence, not a label per GPS point. No evaluator data or route information is used to generate features.

Distances use a trip-local azimuthal equidistant projection in metres. Candidate search uses exact point-to-straight-segment distance within a configurable radius; arc geometry is approximated by the straight line between its supplied endpoints. Radius and candidate limit are initial settings, not optimized values.


**Extended version:** Adds acceleration, turning, road endpoint/projection and directed-degree features, route diagnostics and group-aware CV fold assignments. Existing raw-data input and candidate generation are retained.

In [5]:
import zipfile
from pathlib import Path

zip_path = Path('/content/Student folder.zip')
if zip_path.is_file():
    print(f'Extracting {zip_path.name}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print('Extraction complete!')
else:
    print('Dataset zip file not found.')

Dataset zip file not found.


In [6]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import shapely
from shapely import STRtree
from pyproj import CRS, Transformer
from IPython.display import display

# Locate the dataset on the local Windows machine.
search_roots = [Path.cwd(), Path.home() / "Desktop"]
DATA_DIR = next(
    (p for root in search_roots
     for p in [root / "Student folder", root / "Student folder" / "train"]
     if p.exists()),
    None
)

if DATA_DIR is not None and DATA_DIR.name == "train":
    DATA_DIR = DATA_DIR.parent

if DATA_DIR is None:
    zip_candidates = [
        root / "Student folder.zip"
        for root in search_roots
        if (root / "Student folder.zip").is_file()
    ]
    if not zip_candidates:
        raise FileNotFoundError(
            "Place 'Student folder.zip' or the extracted 'Student folder' "
            "in the notebook directory or Desktop."
        )

    zip_path = zip_candidates[0]
    DATA_DIR = zip_path.parent / "Student folder"

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(zip_path.parent)

OUTPUT_DIR = Path.cwd() / "preprocessing_outputs"
RADIUS_M = 100.0
MAX_CANDIDATES = 8
GAP_THRESHOLD_S = 3.0
MAX_TRIPS_PER_PARTITION = None

assert DATA_DIR.is_dir(), f"Dataset directory not found: {DATA_DIR}"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Dataset:", DATA_DIR.resolve())
print("Output:", OUTPUT_DIR.resolve())

Dataset: C:\Users\sanja\Desktop\PESU\sem-5\ML\hackathon\Student folder
Output: C:\Users\sanja\Desktop\PESU\sem-5\ML\hackathon\preprocessing_outputs


## 1. Load and validate raw data
Rows keep their original zero-based IDs. Invalid GPS rows are excluded from feature extraction and counted; duplicate rows and nonpositive timestamps are flagged rather than silently removed or reordered. Invalid arcs are excluded and counted. No imputation, fitted scaling, reference data, or future observations are used.

In [7]:
def read_table(path, names):
    df = pd.read_csv(path, sep=r'\s+', header=None)
    if df.shape[1] != len(names):
        raise ValueError(f'{path}: expected {len(names)} columns, got {df.shape[1]}')
    df.columns = names
    return df.apply(pd.to_numeric, errors='coerce')

def load_inputs(folder):
    tid = folder.name
    track = read_table(folder / f'{tid}.track', ['longitude', 'latitude', 'timestamp'])
    nodes = read_table(folder / f'{tid}.nodes', ['longitude', 'latitude'])
    arcs = read_table(folder / f'{tid}.arcs', ['from_node_id', 'to_node_id'])
    return track, nodes, arcs

def valid_coords(df):
    return (np.isfinite(df.longitude) & np.isfinite(df.latitude) &
            df.longitude.between(-180, 180) & df.latitude.between(-90, 90))

def prepare_trip(folder):
    track, nodes, arcs = load_inputs(folder)
    track.insert(0, 'observation_id', np.arange(len(track)))
    track['duplicate_gps_row'] = track[['longitude', 'latitude', 'timestamp']].duplicated()
    valid_gps = valid_coords(track) & np.isfinite(track.timestamp)
    gps = track.loc[valid_gps].copy().reset_index(drop=True)
    if gps.empty:
        raise ValueError('No valid GPS observations')
    valid_nodes = valid_coords(nodes).to_numpy()
    raw = arcs[['from_node_id', 'to_node_id']].to_numpy(dtype=float)
    good = np.isfinite(raw).all(axis=1) & (raw == np.floor(raw)).all(axis=1)
    good &= ((raw >= 0) & (raw < len(nodes))).all(axis=1)
    positions = np.flatnonzero(good)
    refs = raw[positions].astype(np.int64)
    good[positions] &= valid_nodes[refs[:, 0]] & valid_nodes[refs[:, 1]]
    valid_arcs = arcs.loc[good].copy()
    valid_arcs.insert(0, 'arc_id', valid_arcs.index.to_numpy())
    valid_arcs[['from_node_id', 'to_node_id']] = valid_arcs[['from_node_id', 'to_node_id']].astype('int64')
    if valid_arcs.empty:
        raise ValueError('No valid road arcs')
    stats = {'raw_observations': len(track), 'invalid_observations': int((~valid_gps).sum()),
             'duplicate_observations': int(track.duplicate_gps_row.sum()),
             'raw_nodes': len(nodes), 'invalid_nodes': int((~valid_nodes).sum()),
             'raw_arcs': len(arcs), 'invalid_arcs': int((~good).sum())}
    return gps, nodes, valid_arcs, stats

## 2. Observation and candidate features
Movement features use only the immediately preceding valid original row and a positive time interval. A missing value means the feature cannot be calculated; it is not replaced with zero. Candidate rank is by distance, then arc ID for deterministic ties. Nonpositive intervals, large gaps and missing candidates are recorded explicitly.


In [8]:
def extract_features(folder, partition):
    gps, nodes, arcs, stats = prepare_trip(folder)
    lat0 = float(gps.latitude.median()); lon0 = float(gps.longitude.median())
    crs = CRS.from_proj4(f'+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m')
    transform = Transformer.from_crs('EPSG:4326', crs, always_xy=True)
    gx, gy = transform.transform(gps.longitude.to_numpy(), gps.latitude.to_numpy())
    gx, gy = np.asarray(gx), np.asarray(gy)
    if not (np.isfinite(gx).all() and np.isfinite(gy).all()):
        raise ValueError('GPS projection failed')
    previous_is_adjacent = gps.observation_id.diff().eq(1).to_numpy()
    dt = gps.timestamp.diff().to_numpy()
    dx = np.r_[np.nan, np.diff(gx)]; dy = np.r_[np.nan, np.diff(gy)]
    step = np.hypot(dx, dy)
    movement_valid = previous_is_adjacent & (dt > 0) & np.isfinite(dt)
    moving = movement_valid & (step > 0)
    gps['dt_s'] = np.where(previous_is_adjacent, dt, np.nan)
    gps['nonpositive_dt'] = previous_is_adjacent & (dt <= 0)
    gps['large_gap'] = movement_valid & (dt > GAP_THRESHOLD_S)
    gps['step_distance_m'] = np.where(movement_valid, step, np.nan)
    gps['speed_mps'] = np.divide(step, dt, out=np.full(len(gps), np.nan), where=movement_valid)
    gps['heading_deg'] = np.where(moving, (np.degrees(np.arctan2(dx, dy)) + 360) % 360, np.nan)

    from_ids = arcs.from_node_id.to_numpy(); to_ids = arcs.to_node_id.to_numpy()
    lon = nodes.longitude.to_numpy(); lat = nodes.latitude.to_numpy()
    ux, uy = transform.transform(lon[from_ids], lat[from_ids])
    vx, vy = transform.transform(lon[to_ids], lat[to_ids])
    ux, uy, vx, vy = map(np.asarray, (ux, uy, vx, vy))
    finite = np.isfinite(ux) & np.isfinite(uy) & np.isfinite(vx) & np.isfinite(vy)
    stats['unprojectable_arcs'] = int((~finite).sum())
    arcs = arcs.loc[finite].reset_index(drop=True)
    ux, uy, vx, vy = ux[finite], uy[finite], vx[finite], vy[finite]
    if not len(arcs):
        raise ValueError('No projectable road arcs')
    lines = shapely.linestrings(np.stack([np.column_stack([ux, uy]), np.column_stack([vx, vy])], axis=1))
    points = shapely.points(gx, gy)
    pairs = STRtree(lines).query(points, predicate='dwithin', distance=RADIUS_M)
    pi, ai = pairs[0], pairs[1]
    distance = shapely.distance(points[pi], lines[ai])
    arc_ids = arcs.arc_id.to_numpy()[ai]
    order = np.lexsort((arc_ids, distance, pi))
    pi, ai, distance = pi[order], ai[order], distance[order]
    ranks = pd.Series(pi).groupby(pi, sort=False).cumcount().to_numpy() + 1
    keep = ranks <= MAX_CANDIDATES
    pi, ai, distance, ranks = pi[keep], ai[keep], distance[keep], ranks[keep]
    edge_dx = vx[ai] - ux[ai]; edge_dy = vy[ai] - uy[ai]
    length = np.hypot(edge_dx, edge_dy)
    bearing = np.where(length > 0, (np.degrees(np.arctan2(edge_dx, edge_dy)) + 360) % 360, np.nan)
    heading = gps.heading_deg.to_numpy()[pi]
    angle = np.abs((heading - bearing + 180) % 360 - 180)
    # Candidate geometry: position along each directed arc and endpoint distances.
    safe_len_sq = np.maximum(length ** 2, 1e-12)
    fraction = np.clip(((gx[pi] - ux[ai]) * edge_dx + (gy[pi] - uy[ai]) * edge_dy) / safe_len_sq, 0, 1)
    fraction = np.where(length > 0, fraction, 0.0)
    distance_to_start = np.hypot(gx[pi] - ux[ai], gy[pi] - uy[ai])
    distance_to_end = np.hypot(gx[pi] - vx[ai], gy[pi] - vy[ai])
    snapped_x = ux[ai] + fraction * edge_dx
    snapped_y = uy[ai] + fraction * edge_dy
    # Directed node degrees; parallel arcs are retained.
    indegree = np.bincount(arcs.to_node_id.to_numpy(), minlength=len(nodes))
    outdegree = np.bincount(arcs.from_node_id.to_numpy(), minlength=len(nodes))
    # Movement changes require consecutive, positive-time observations.
    speed = gps.speed_mps.to_numpy()
    accel = np.full(len(gps), np.nan)
    valid_accel = movement_valid & np.r_[False, movement_valid[:-1]] & np.isfinite(speed) & np.isfinite(np.r_[np.nan, speed[:-1]])
    accel[valid_accel] = (speed[valid_accel] - np.r_[np.nan, speed[:-1]][valid_accel]) / dt[valid_accel]
    hd = gps.heading_deg.to_numpy()
    turn = np.full(len(gps), np.nan)
    turn_ok = moving & np.r_[False, moving[:-1]] & np.isfinite(hd) & np.isfinite(np.r_[np.nan, hd[:-1]])
    turn[turn_ok] = np.abs((hd[turn_ok] - np.r_[np.nan, hd[:-1]][turn_ok] + 180) % 360 - 180)
    gps['accel_ms2'] = accel
    gps['turn_abs_deg'] = turn
    gps['speed_kmh'] = gps.speed_mps * 3.6
    gps['x_m'] = gx
    gps['y_m'] = gy
    candidates = pd.DataFrame({'partition': partition, 'trip_id': folder.name,
        'observation_id': gps.observation_id.to_numpy()[pi], 'arc_id': arcs.arc_id.to_numpy()[ai],
        'from_node_id': arcs.from_node_id.to_numpy()[ai], 'to_node_id': arcs.to_node_id.to_numpy()[ai],
        'candidate_rank': ranks, 'distance_to_arc_m': distance, 'arc_length_m': length,
        'arc_bearing_deg': bearing, 'heading_difference_deg': angle,
        'projection_fraction': fraction, 'distance_to_start_m': distance_to_start,
        'distance_to_end_m': distance_to_end, 'snapped_x_m': snapped_x, 'snapped_y_m': snapped_y,
        'from_in_degree': indegree[arcs.from_node_id.to_numpy()[ai]],
        'from_out_degree': outdegree[arcs.from_node_id.to_numpy()[ai]],
        'to_in_degree': indegree[arcs.to_node_id.to_numpy()[ai]],
        'to_out_degree': outdegree[arcs.to_node_id.to_numpy()[ai]]})
    counts = np.bincount(pi, minlength=len(gps))
    gps.insert(0, 'trip_id', folder.name); gps.insert(0, 'partition', partition)
    gps['candidate_count'] = counts
    stats.update(valid_observations=len(gps), observations_without_candidates=int((counts == 0).sum()),
                 candidate_rows=len(candidates), nonpositive_intervals=int(gps.nonpositive_dt.sum()),
                 gaps_over_threshold=int(gps.large_gap.sum()))
    return gps, candidates, stats

## 3. Process available trips and save outputs
Each trip is saved separately as Parquet to limit memory use. A manifest records success and failures. The `observation_id`, `arc_id`, trip and partition columns preserve joins; arc IDs are local to each trip. The candidate table intentionally has no target label. Existing files in `preprocessing_outputs/` for the same trip are overwritten on rerun.

In [9]:
records = []
started = time.time()
for partition in ['train', 'student_test']:
    root = DATA_DIR / partition
    if not root.is_dir():
        raise FileNotFoundError(root)
    folders = sorted(p for p in root.iterdir() if p.is_dir())
    if MAX_TRIPS_PER_PARTITION is not None:
        folders = folders[:MAX_TRIPS_PER_PARTITION]
    destination = OUTPUT_DIR / partition
    destination.mkdir(exist_ok=True)
    for i, folder in enumerate(folders, 1):
        record = {'partition': partition, 'trip_id': folder.name}
        try:
            gps, candidates, stats = extract_features(folder, partition)
            gps.to_parquet(destination / f'{folder.name}_observations.parquet', index=False)
            candidates.to_parquet(destination / f'{folder.name}_candidates.parquet', index=False)
            record.update(stats, status='ok', error='')
        except Exception as exc:
            record.update(status='error', error=f'{type(exc).__name__}: {exc}')
            print('FAILED:', partition, folder.name, record['error'])
        records.append(record)
        if i % 10 == 0 or i == len(folders):
            print(f'{partition}: {i}/{len(folders)} trips; {time.time()-started:.1f}s elapsed')
manifest = pd.DataFrame(records)
manifest.to_csv(OUTPUT_DIR / 'processing_manifest.csv', index=False)
config = {'radius_m': RADIUS_M, 'max_candidates': MAX_CANDIDATES, 'gap_threshold_s': GAP_THRESHOLD_S,
          'max_trips_per_partition': MAX_TRIPS_PER_PARTITION, 'projection': 'trip-local azimuthal equidistant',
          'arc_geometry': 'straight line between supplied endpoints'}
(OUTPUT_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print('Saved manifest and configuration.')

train: 10/80 trips; 1.8s elapsed
train: 20/80 trips; 4.5s elapsed
train: 30/80 trips; 7.3s elapsed
train: 40/80 trips; 9.3s elapsed
train: 50/80 trips; 11.8s elapsed
train: 60/80 trips; 13.2s elapsed
train: 70/80 trips; 16.0s elapsed
train: 80/80 trips; 18.3s elapsed
student_test: 10/10 trips; 19.6s elapsed
Saved manifest and configuration.


## 4. Results and next step
Candidate coverage below means at least one road was found within the configured radius after validation. **It is not reference candidate recall or model accuracy**, since observation-level reference alignment is not yet available.

In [10]:
ok = manifest.loc[manifest.status.eq('ok')].copy()
failed = manifest.loc[manifest.status.ne('ok'), ['partition', 'trip_id', 'error']]
metrics = ['raw_observations', 'valid_observations', 'invalid_observations', 'duplicate_observations',
           'raw_nodes', 'invalid_nodes', 'raw_arcs', 'invalid_arcs', 'unprojectable_arcs',
           'nonpositive_intervals', 'gaps_over_threshold', 'observations_without_candidates', 'candidate_rows']
if len(ok):
    summary = ok.groupby('partition')[metrics].sum().astype('int64')
    summary.insert(0, 'trips_processed', ok.groupby('partition').size())
    summary['candidate_coverage_pct'] = 100 * (1 - summary.observations_without_candidates / summary.valid_observations)
    summary['candidates_per_observation'] = summary.candidate_rows / summary.valid_observations
    display(summary.T)
    summary.to_csv(OUTPUT_DIR / 'processing_summary.csv')
    first = ok.iloc[0]
    first_dir = OUTPUT_DIR / first['partition']
    print('Example observation features:')
    display(pd.read_parquet(first_dir / f"{first['trip_id']}_observations.parquet").head())
    print('Example candidate features:')
    display(pd.read_parquet(first_dir / f"{first['trip_id']}_candidates.parquet").head())
print('Failed trips:', len(failed))
if len(failed): display(failed)
print('Next: validate reference-route-to-observation alignment before producing labels, candidate miss rate or training models.')

partition,student_test,train
trips_processed,10.000000,8.000000e+01
raw_observations,19160.000000,2.084170e+05
valid_observations,19160.000000,2.084170e+05
invalid_observations,0.000000,0.000000e+00
duplicate_observations,0.000000,0.000000e+00
raw_nodes,90689.000000,1.933485e+06
invalid_nodes,0.000000,0.000000e+00
raw_arcs,185041.000000,3.876904e+06
invalid_arcs,0.000000,0.000000e+00
unprojectable_arcs,0.000000,0.000000e+00


Example observation features:


,partition,trip_id,observation_id,longitude,latitude,timestamp,duplicate_gps_row,dt_s,nonpositive_dt,large_gap,step_distance_m,speed_mps,heading_deg,accel_ms2,turn_abs_deg,speed_kmh,x_m,y_m,candidate_count
0,train,00000001,0,8.595203,49.844053,1.0,False,NaN,False,False,NaN,NaN,NaN,NaN,NaN,NaN,-601.387443,-31615.528428,8
1,train,00000001,1,8.595294,49.844441,2.0,False,1.0,False,False,43.649256,43.649256,8.630643,NaN,NaN,157.137321,-594.837256,-31572.373444,5
2,train,00000001,2,8.595352,49.844883,3.0,False,1.0,False,False,49.338606,49.338606,4.856659,5.689351,3.773983,177.618983,-590.660090,-31523.211982,6
3,train,00000001,3,8.595390,49.845379,4.0,False,1.0,False,False,55.235798,55.235798,2.842553,5.897191,2.014107,198.848873,-587.920861,-31468.044147,6
4,train,00000001,4,8.595587,49.845750,5.0,False,1.0,False,False,43.629800,43.629800,18.957354,-11.605998,16.114801,157.067279,-573.747096,-31426.780800,8


Example candidate features:


,partition,trip_id,observation_id,arc_id,from_node_id,to_node_id,candidate_rank,distance_to_arc_m,arc_length_m,arc_bearing_deg,heading_difference_deg,projection_fraction,distance_to_start_m,distance_to_end_m,snapped_x_m,snapped_y_m,from_in_degree,from_out_degree,to_in_degree,to_out_degree
0,train,00000001,0,64973,32308,70987,1,0.723049,293.352246,7.317066,NaN,0.109775,32.210996,261.150367,-600.670282,-31615.620515,1,1,1,1
1,train,00000001,0,117639,61984,61983,2,15.138380,1020.835864,187.386430,NaN,0.512692,523.593313,497.691728,-616.400199,-31613.582226,1,1,1,1
2,train,00000001,0,2641,1584,32308,3,32.210996,425.215971,7.188148,NaN,1.000000,457.420858,32.210996,-604.771643,-31647.561152,1,1,1,1
3,train,00000001,0,50652,24964,26709,4,93.046268,169.107907,176.786956,NaN,0.845208,170.549094,96.658251,-508.487441,-31610.313295,2,2,3,3
4,train,00000001,0,53932,26709,24964,5,93.046268,169.107907,356.786956,NaN,0.154792,96.658251,170.549094,-508.487441,-31610.313295,3,3,2,2


Failed trips: 0
Next: validate reference-route-to-observation alignment before producing labels, candidate miss rate or training models.


## 5. Additional validation, route diagnostics and trip-level CV folds

The `.route` file is a chronological sequence of arc IDs, **not an observation-level label**. The following route checks are diagnostics only. They are **not joined to candidate features**. CV folds group by trip and are saved only for the training partition. Any learned scaling must be fitted separately inside each training fold (not globally before validation).

In [11]:
from sklearn.model_selection import GroupKFold

# Check feature schema and geometry on one successfully processed trip.
if len(ok):
    first = ok.iloc[0]
    root = OUTPUT_DIR / first.partition
    obs = pd.read_parquet(root / f'{first.trip_id}_observations.parquet')
    cand = pd.read_parquet(root / f'{first.trip_id}_candidates.parquet')
    expected_obs = ['accel_ms2', 'turn_abs_deg', 'speed_kmh', 'x_m', 'y_m']
    expected_cand = ['projection_fraction', 'distance_to_start_m', 'distance_to_end_m',
                     'snapped_x_m', 'snapped_y_m', 'from_in_degree', 'from_out_degree',
                     'to_in_degree', 'to_out_degree']
    assert set(expected_obs).issubset(obs.columns)
    assert set(expected_cand).issubset(cand.columns)
    assert cand.projection_fraction.between(0, 1).all()
    assert (cand.distance_to_arc_m >= 0).all()
    assert not cand.duplicated(['partition','trip_id','observation_id','arc_id']).any()
    print('Feature validation passed on', first.partition, first.trip_id)

# The arc IDs in .route refer to row positions in the original .arcs file.
route_checks = []
for partition in ['train', 'student_test']:
    base = DATA_DIR / partition
    for folder in sorted(p for p in base.iterdir() if p.is_dir()):
        if MAX_TRIPS_PER_PARTITION is not None and folder.name not in set(manifest.loc[manifest.partition.eq(partition), 'trip_id']):
            continue
        tid = folder.name
        path = folder / f'{tid}.route'
        if not path.exists():
            route_checks.append({'partition':partition, 'trip_id':tid, 'route_available':False})
            continue
        route = pd.read_csv(path, sep=r'\s+', header=None).iloc[:, 0]
        arc_path = folder / f'{tid}.arcs'
        arcs_raw = read_table(arc_path, ['from_node_id', 'to_node_id'])
        ids = pd.to_numeric(route, errors='coerce').to_numpy(dtype=float)
        valid = np.isfinite(ids) & (ids == np.floor(ids)) & (ids >= 0) & (ids < len(arcs_raw))
        seq = ids[valid].astype(int)
        valid_edges = arcs_raw.iloc[seq]
        u = valid_edges.from_node_id.to_numpy()
        v = valid_edges.to_node_id.to_numpy()
        connected = np.isfinite(v[:-1]) & np.isfinite(u[1:]) & (v[:-1] == u[1:])
        route_checks.append({'partition':partition, 'trip_id':tid, 'route_available':True,
                             'route_rows':len(ids), 'invalid_route_arc_ids':int((~valid).sum()),
                             'disconnected_consecutive_arc_pairs':int((~connected).sum())})
route_report = pd.DataFrame(route_checks)
route_report.to_csv(OUTPUT_DIR / 'route_diagnostics.csv', index=False)
display(route_report.groupby('partition').sum(numeric_only=True))

train_ids = manifest.loc[(manifest.partition == 'train') & (manifest.status == 'ok'), 'trip_id'].drop_duplicates().sort_values().to_numpy()
if len(train_ids) >= 2:
    n_folds = min(5, len(train_ids))
    fold_ids = np.empty(len(train_ids), dtype=int)
    splitter = GroupKFold(n_splits=n_folds)
    for fold, (_, valid_idx) in enumerate(splitter.split(train_ids, groups=train_ids)):
        fold_ids[valid_idx] = fold
    cv_folds = pd.DataFrame({'partition':'train', 'trip_id':train_ids, 'fold':fold_ids})
    cv_folds.to_csv(OUTPUT_DIR / 'cv_folds.csv', index=False)
    print('Saved', n_folds, 'grouped CV folds to', OUTPUT_DIR / 'cv_folds.csv')
else:
    print('Need at least 2 successful training trips for CV folds.')
print('No target labels or route-derived features were added to candidate tables.')

Feature validation passed on train 00000001


,route_available,route_rows,invalid_route_arc_ids,disconnected_consecutive_arc_pairs
partition,,,,
student_test,10,3477,0,1
train,80,31336,0,13


Saved 5 grouped CV folds to c:\Users\sanja\Desktop\PESU\sem-5\ML\hackathon\preprocessing_outputs\cv_folds.csv
No target labels or route-derived features were added to candidate tables.
